# 01 - Preparacao dos Dados

Alvo `ltfu`: **1 = abandono** (`SITUA_ENCE=2`), **0 = cura** (`SITUA_ENCE=1`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.options.display.float_format = '{:.3f}'.format
import sys; sys.path.append("../src")   
DATA = "../data/processed"


## Da base bruta aos conjuntos de treino/teste
O script `src/data_prep.py` (em polars) recebe os microdados do SINAN
(`tuberculose_unificado.feather`) e gera `treino.csv`, `teste1.csv`, `teste2.csv`.
Decisoes: adultos (>=18), TB pulmonar, casos sensiveis; alvo a partir de
`SITUA_ENCE`; **divisao temporal** (treino < 2025; teste = 2025 dividido ao meio).

In [ ]:
treino = pd.read_csv(f"{DATA}/treino.csv")
teste1 = pd.read_csv(f"{DATA}/teste1.csv")
teste2 = pd.read_csv(f"{DATA}/teste2.csv")

for nome, dt in [("treino", treino), ("teste1", teste1), ("teste2", teste2)]:
    print(f"{nome}: {dt.shape[0]} linhas | abandono = {dt['ltfu'].mean():.1%}")

**Observacao:** a taxa de abandono e maior em 2025 (teste) do que no treino.
Isso ocorre por *censura a direita*: casos de cura so sao encerrados apos ~6 meses,
entao em 2025 os encerramentos disponiveis sao mais abandonos. (Discutido no relatorio.)

## Pre-processamento

In [ ]:
# listas de variaveis e o pre-processamento
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num = ["idade_anos", "NU_CONTATO", "dias_diag_trat"]
cat = ["CS_SEXO","CS_RACA","CS_ESCOL_N","SG_UF_NOT","RAIOX_TORA","TESTE_TUBE",
       "BACILOSC_E","HIV","AGRAVAIDS","AGRAVALCOO","AGRAVDIABE","AGRAVDOENC",
       "AGRAVDROGA","AGRAVTABAC","TRAT_SUPER","ANT_RETRO","BENEF_GOV",
       "POP_RUA","POP_LIBER","POP_IMIG","POP_SAUDE"]

def preparar(df):
    # cria o atraso ate o inicio do tratamento e deixa as categoricas como texto
    df = df.copy()
    df["DT_DIAG"]    = pd.to_datetime(df["DT_DIAG"], errors="coerce")
    df["DT_INIC_TR"] = pd.to_datetime(df["DT_INIC_TR"], errors="coerce")
    df["dias_diag_trat"] = (df["DT_INIC_TR"] - df["DT_DIAG"]).dt.days
    for c in cat:
        df[c] = df[c].fillna("ignorado").astype(str)
    return df[num + cat]
print("Numericas :", num)
print("Categoricas:", cat)

In [ ]:
# preparar() cria o atraso ate o tratamento e ajeita as categoricas
X_treino = preparar(treino)
X_treino.head(3)

In [ ]:
# normaliza as numericas e faz one-hot nas categoricas
transformadores = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat),
], verbose_feature_names_out=False)
X_treino_prep = transformadores.fit_transform(X_treino)
print("Matriz pronta para o modelo:", X_treino_prep.shape)
print("Exemplos de colunas:", list(transformadores.get_feature_names_out())[:6], "...")

### Resumo
- **Idade** decodificada para anos (mantidos >= 18).
- **`dias_diag_trat`**: atraso entre diagnostico e inicio do tratamento.
- **Missing numerico** -> mediana; **missing categorico** -> categoria `ignorado`.
- **Normalizacao** (`StandardScaler`) e **codificacao** (`OneHotEncoder`).
- Removida `SITUA_ENCE` (origem do alvo) para evitar vazamento.